# Inventory Health Analytics
**Summit & Stone Outfitters — Fictional Retailer (~$150M revenue)**

Identifies underperforming inventory across departments using a five-layer
diagnostic framework: regression residuals, dispersion, tail drag, sell-through
with markdown overlay, and on-order compounding.

All data is synthetic. See `synthetic_data_gen.py` to regenerate.


## Cell 1 — Setup & Data Load

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import OLSInfluence

pd.set_option("display.float_format", "{:,.3f}".format)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False,
                     "axes.spines.right": False})

# ── Load data ──────────────────────────────────────────────────────────────
dept    = pd.read_csv("data/department_summary.csv")
skus    = pd.read_csv("data/sku_detail.csv")
cadence = pd.read_csv("data/sale_cadence.csv")
pos     = pd.read_csv("data/open_purchase_orders.csv")

# ── Merge cadence and PO data into SKU detail ──────────────────────────────
skus = skus.merge(
    cadence[["SKUID", "TxnCount", "AvgDaysBetweenSales",
             "MaxGapDays", "DaysSinceLastSale"]],
    on="SKUID", how="left"
)

po_by_sku = (pos.groupby("SKUID")
             .agg(UnitsOnOrder=("UnitsOnOrder", "sum"),
                  POCostValue=("POCostValue",  "sum"),
                  ExpectedReceiptDays=("ExpectedReceiptDays", "min"))
             .reset_index())
skus = skus.merge(po_by_sku, on="SKUID", how="left")
skus["HasOpenPO"] = skus["UnitsOnOrder"].notna()

# ── Log-transformed features for regression ───────────────────────────────
dept["LogTurns"]       = np.log(dept["InventoryTurns"].clip(lower=0.01))
dept["LogMarkupRatio"] = np.log(dept["MarkupRatio"].clip(lower=0.01))
dept["LogRevenue"]     = np.log(dept["NetBookedRevenue"].clip(lower=1))
dept["LogPriorSTR"]    = np.log(dept["PriorYearSTR"].clip(lower=0.001))

print(f"Departments : {len(dept)}")
print(f"Total SKUs  : {len(skus):,}")
print(f"Open PO lines: {len(pos):,}")
print(f"\n{'Department':<28} {'Turns':>6} {'PriorSTR':>9} {'MarkupRatio':>12} {'SKUCount':>9}")
print("-" * 68)
for _, r in dept.sort_values("InventoryTurns").iterrows():
    print(f"  {r['Department']:<26} {r['InventoryTurns']:>6.2f} "
          f"{r['PriorYearSTR']:>9.3f} {r['MarkupRatio']:>12.3f} {int(r['SKUCount']):>9}")

## Cell 2 — Turns Distributions

Department-level bar chart and SKU-level histogram. The department chart
shows where capital is concentrated relative to velocity. The SKU histogram
shows the full distribution — aggregate turns smooth over the long tail of
slow or non-moving SKUs within each department.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Left: department turns bar, sorted ---
dept_s = dept.sort_values("InventoryTurns")
bar_colors = ["#d9534f" if t < 1.5 else "#f0ad4e" if t < 2.5 else "#5cb85c"
              for t in dept_s["InventoryTurns"]]
axes[0].barh(dept_s["Department"], dept_s["InventoryTurns"],
             color=bar_colors, edgecolor="white", height=0.7)
axes[0].axvline(dept_s["InventoryTurns"].median(), color="#555",
                linestyle="--", linewidth=1, label=f"Median: {dept_s['InventoryTurns'].median():.2f}")
axes[0].set_xlabel("Inventory Turns")
axes[0].set_title("Inventory Turns by Department")
axes[0].legend(fontsize=9)
for spine in ["top", "right"]:
    axes[0].spines[spine].set_visible(False)

# --- Right: SKU-level log turns histogram ---
log_turns = np.log(skus["ItemTurns"].clip(0.01))
axes[1].hist(log_turns, bins=55, color="#4a90d9", edgecolor="white", alpha=0.85)
axes[1].set_xlabel("log(Item Turns)")
axes[1].set_ylabel("SKU Count")
axes[1].set_title("SKU-Level Turns Distribution (log scale)")
for pct, lbl in [(25, "P25"), (50, "P50"), (75, "P75")]:
    v = np.percentile(log_turns.dropna(), pct)
    axes[1].axvline(v, color="#555", linestyle=":", linewidth=1)
    axes[1].text(v + 0.05, axes[1].get_ylim()[1] * 0.90, lbl, fontsize=8, color="#555")

plt.tight_layout()
plt.show()

print(f"\nSKU turns percentiles:")
for p in [10, 25, 50, 75, 90]:
    print(f"  P{p:<3}: {np.percentile(skus['ItemTurns'], p):.3f}")
print(f"\nSKUs with zero sales history: {skus['NoSalesHistory'].sum():,} "
      f"({skus['NoSalesHistory'].mean()*100:.1f}%)")

## Cell 3 — Velocity Cadence: Avg Days Between Sales & Max Gap

Aggregate turns tell you *how much* sold relative to inventory.
Cadence tells you *how consistently* it sold. A department with turns of 1.5
that sells steadily is healthier than one with turns of 1.5 from a burst of
sales followed by months of inactivity.

Cadence metrics are rolled up from SKU level to department level here.
SKU-level cadence is used in Cell 13 (Item-Level Diagnostic) and Cell 14
(Item Velocity Flags).


In [ ]:
# ── Dept-level cadence rollup from SKU detail ─────────────────────────────
# Exclude no-history SKUs from cadence aggregation
cadence_skus = skus[skus["TxnCount"] > 0].copy()

dept_cadence = (
    cadence_skus.groupby("Department")
    .agg(
        MedianAvgGap    =("AvgDaysBetweenSales", "median"),
        MedianMaxGap    =("MaxGapDays",          "median"),
        MedianDaysSince =("DaysSinceLastSale",   "median"),
        NoHistorySKUs   =("TxnCount",            lambda x: (x == 0).sum()),
    )
    .reset_index()
)

dept = dept.merge(dept_cadence, on="Department", how="left")

print("Dept-level cadence (medians across SKUs, excl. no-history):")
print(dept[["Department", "InventoryTurns", "MedianAvgGap",
            "MedianMaxGap", "MedianDaysSince"]]
      .sort_values("MedianMaxGap", ascending=False)
      .to_string(index=False))

# ── Plot: Max gap vs turns ─────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(dept["InventoryTurns"], dept["MedianMaxGap"],
           s=dept["AvgDailyInvCOGS"] / 8000, color="#4a90d9",
           edgecolors="white", linewidths=0.5, alpha=0.85)
for _, r in dept.iterrows():
    ax.annotate(r["Department"], (r["InventoryTurns"], r["MedianMaxGap"]),
                fontsize=6.5, xytext=(4, 2), textcoords="offset points", color="#444")
ax.set_xlabel("Inventory Turns")
ax.set_ylabel("Median Max Gap Between Sales (days)")
ax.set_title("Sale Consistency vs. Turns\n(bubble size = avg daily inventory)")

ax = axes[1]
ax.scatter(dept["MedianAvgGap"], dept["MedianDaysSince"],
           s=dept["AvgDailyInvCOGS"] / 8000, color="#e8834a",
           edgecolors="white", linewidths=0.5, alpha=0.85)
for _, r in dept.iterrows():
    ax.annotate(r["Department"], (r["MedianAvgGap"], r["MedianDaysSince"]),
                fontsize=6.5, xytext=(4, 2), textcoords="offset points", color="#444")
ax.set_xlabel("Median Avg Days Between Sales")
ax.set_ylabel("Median Days Since Last Sale")
ax.set_title("Sale Recency vs. Cadence\n(bubble size = avg daily inventory)")

plt.tight_layout()
plt.show()

## Cell 4 — OLS Regression

**Model:** `log(InventoryTurns) ~ log(PriorYearSTR) + log(MarkupRatio) + SeasonalityCV + log(SKUCount)`

`PriorYearSTR` is a 3-year average sell-through rate — the single strongest
structural predictor. A department that sold through well historically is
expected to do so again. Controlling for it means the residual captures
*current-year underperformance relative to that department's own baseline*,
not just low turns in absolute terms.

Departments with standardized residual < −1.0 are flagged for the regression
watchlist.


In [ ]:
df_reg = dept.dropna(subset=["LogPriorSTR", "LogMarkupRatio",
                              "SeasonalityCV", "LogRevenue", "LogTurns"]).copy()

X   = sm.add_constant(df_reg[["LogPriorSTR", "LogMarkupRatio",
                               "SeasonalityCV", "LogRevenue"]])
y   = df_reg["LogTurns"]
fit = sm.OLS(y, X).fit()

print(fit.summary())

df_reg["PredictedTurns"] = np.exp(fit.fittedvalues)
df_reg["Residual"]       = fit.resid
df_reg["ResidualStd"]    = (fit.resid / np.sqrt(fit.mse_resid)).round(4)

# Write back to dept so downstream cells have it
dept = dept.merge(
    df_reg[["Department", "PredictedTurns", "Residual", "ResidualStd"]],
    on="Department", how="left"
)

print("\nActual vs. predicted turns:")
print(dept[["Department", "InventoryTurns", "PredictedTurns", "ResidualStd"]]
      .sort_values("ResidualStd")
      .to_string(index=False))

regression_wl = set(dept[dept["ResidualStd"] < -1.0]["Department"])
print(f"\nRegression watchlist ({len(regression_wl)} departments): {sorted(regression_wl)}")

## Cell 5 — Dispersion Watchlist

High within-department CV (> 1.5) with meaningful SKU count and inventory.
Flags departments where turns are spread so unevenly across SKUs that the
aggregate figure is largely meaningless — a small number of fast movers
are masking a large body of stagnant inventory.


In [ ]:
dispersion_wl = dept[
    (dept["TurnsCV"]          > 0.70) &
    (dept["SKUCount"]         >= 20) &
    (dept["AvgDailyInvCOGS"]  >= 10_000)
][["Department", "InventoryTurns", "TurnsCV",
   "TurnsP25", "TurnsP75", "SKUCount", "BrandCount", "AvgDailyInvCOGS"]
].sort_values("TurnsCV", ascending=False).round(3)

print("=" * 60)
print(f"DISPERSION WATCHLIST  (CV > 1.5, SKUs >= 20, Inv >= $10K)")
print("=" * 60)
print(f"Flagged: {len(dispersion_wl)} departments\n")
print(dispersion_wl.to_string(index=False))

dispersion_wl_set = set(dispersion_wl["Department"])

## Cell 6 — Tail Drag Watchlist

Distinct from dispersion: targets departments where the *bottom quartile*
of SKUs is dragging the aggregate down. CV > 0.60 confirms spread exists;
TurnsP25 < 1.05 confirms the slow end is genuinely stagnant, not just
moderately below average.


In [ ]:
tail_drag_wl = dept[
    (dept["TurnsCV"]          > 0.60) &
    (dept["TurnsP25"]         < 1.05) &
    (dept["SKUCount"]         >= 20) &
    (dept["AvgDailyInvCOGS"]  >= 10_000)
][["Department", "InventoryTurns", "TurnsCV",
   "TurnsP25", "TurnsP75", "SKUCount", "BrandCount", "AvgDailyInvCOGS"]
].sort_values("TurnsP25", ascending=True).round(3)

print("=" * 60)
print(f"TAIL DRAG WATCHLIST  (CV > 1.0, P25 < 0.75, SKUs >= 20, Inv >= $10K)")
print("=" * 60)
print(f"Flagged: {len(tail_drag_wl)} departments\n")
print(tail_drag_wl.to_string(index=False))

tail_drag_wl_set = set(tail_drag_wl["Department"])

## Cell 7 — Sell-Through & Markdown

Two views combined:

**Sell-through watchlist** — departments with STR < 0.49 and turns < 1.0.
STR is proxied as `turns / (turns + replenishment_cycles)`. A department
turning at 0.9 receiving 3 replenishment orders per year is only converting
~23% of available inventory into sales.

**Markdown overlay** — of those flagged departments, which are already
marked down at a meaningful rate? High markdown rate + low STR means
discounting isn't clearing the inventory. That changes the buyer
conversation from "try a discount" to "assortment or demand problem."


In [ ]:
REPLEN_CYCLES = 3.0
dept["SellThroughRate"] = (dept["InventoryTurns"] /
                           (dept["InventoryTurns"] + REPLEN_CYCLES)).round(4)

sell_through_wl = dept[
    (dept["SellThroughRate"] < 0.49) &
    (dept["InventoryTurns"]  < 1.0)
][["Department", "InventoryTurns", "SellThroughRate",
   "AvgDailyInvCOGS"]
].sort_values("AvgDailyInvCOGS", ascending=False).round(3)

print("=" * 60)
print("SELL-THROUGH WATCHLIST  (STR < 0.49, Turns < 1.0)")
print("=" * 60)
print(f"Flagged: {len(sell_through_wl)} departments\n")
print(sell_through_wl.to_string(index=False))

sell_through_wl_set = set(sell_through_wl["Department"])

# ── Markdown overlay: dept-level markdown rate from SKU detail ─────────────
dept_markdown = (
    skus.groupby("Department")
    .agg(
        TotalSKUs     =("SKUID",         "count"),
        MarkdownSKUs  =("IsMarkdown",    "sum"),
        AvgMarkdownDepth=("MarkdownDepth","mean"),
        AvgMarkdownRate =("MarkdownRate", "mean"),
    )
    .reset_index()
)
dept_markdown["MarkdownPct"] = (dept_markdown["MarkdownSKUs"] /
                                 dept_markdown["TotalSKUs"]).round(3)

all_wl_so_far = (regression_wl | dispersion_wl_set |
                 tail_drag_wl_set | sell_through_wl_set)

markdown_flagged = dept_markdown[
    (dept_markdown["Department"].isin(all_wl_so_far)) &
    (dept_markdown["MarkdownPct"] >= 0.20)
].merge(
    dept[["Department", "InventoryTurns", "SellThroughRate", "AvgDailyInvCOGS"]],
    on="Department", how="left"
).sort_values("MarkdownPct", ascending=False).round(3)

print("\n" + "=" * 60)
print("MARKDOWN OVERLAY  (Watchlist depts with >= 20% SKUs marked down)")
print("=" * 60)
print(f"Flagged: {len(markdown_flagged)} departments\n")
print(markdown_flagged[["Department", "MarkdownPct", "AvgMarkdownDepth",
                         "InventoryTurns", "SellThroughRate",
                         "AvgDailyInvCOGS"]].to_string(index=False))

# ── Plot: STR vs markdown rate across all departments ──────────────────────
dept_plot = dept.merge(dept_markdown[["Department","MarkdownPct","AvgMarkdownDepth"]],
                       on="Department", how="left")

fig, ax = plt.subplots(figsize=(9, 6))
colors = ["#d9534f" if d in all_wl_so_far else "#4a90d9"
          for d in dept_plot["Department"]]
ax.scatter(dept_plot["SellThroughRate"], dept_plot["MarkdownPct"],
           s=dept_plot["AvgDailyInvCOGS"] / 8000,
           c=colors, edgecolors="white", linewidths=0.5, alpha=0.85)
ax.axvline(0.49, color="#d9534f", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(0.20, color="#f0ad4e", linestyle="--", linewidth=1, alpha=0.7)
for _, r in dept_plot.iterrows():
    ax.annotate(r["Department"], (r["SellThroughRate"], r["MarkdownPct"]),
                fontsize=6.5, xytext=(4, 2), textcoords="offset points", color="#444")
ax.set_xlabel("Sell-Through Rate")
ax.set_ylabel("Markdown SKU %")
ax.set_title("Sell-Through vs. Markdown Rate\n(red = on watchlist, bubble = inventory exposure)")
handles = [mpatches.Patch(color="#d9534f", label="On watchlist"),
           mpatches.Patch(color="#4a90d9", label="Clean")]
ax.legend(handles=handles, fontsize=9)
plt.tight_layout()
plt.show()

## Cell 8 — On-Order Compounding Flag

For departments already on the watchlist, checks whether open purchase
orders are making the problem worse. `OnOrderRatio > 1.0` means more
inventory is inbound than is currently on hand — a department that can't
sell what it has is about to receive more of it.

This is a compounding condition, not a standalone flag. It doesn't add
a department to the watchlist; it escalates departments already on it.


In [ ]:
# ── Dept-level PO rollup ───────────────────────────────────────────────────
dept_pos = (
    pos.groupby("Department")
    .agg(TotalOnOrderCOGS=("POCostValue", "sum"),
         TotalOnOrderUnits=("UnitsOnOrder","sum"),
         OpenPOLines      =("SKUID",      "count"))
    .reset_index()
)

dept = dept.merge(dept_pos, on="Department", how="left")
dept["TotalOnOrderCOGS"]  = dept["TotalOnOrderCOGS"].fillna(0)
dept["TotalOnOrderUnits"] = dept["TotalOnOrderUnits"].fillna(0)

# OnOrderRatio already in dept summary; recalculate for clarity
dept["OnOrderRatio"] = (dept["TotalOnOrderCOGS"] /
                        dept["AvgDailyInvCOGS"].clip(1)).round(4)

compounding = dept[
    (dept["Department"].isin(all_wl_so_far)) &
    (dept["OnOrderRatio"] > 1.0)
][["Department", "InventoryTurns", "SellThroughRate",
   "TotalOnOrderCOGS", "OnOrderRatio", "AvgDailyInvCOGS"]
].sort_values("OnOrderRatio", ascending=False).round(3)

oor_wl_set = set(compounding["Department"])

print("=" * 60)
print("ON-ORDER COMPOUNDING FLAG  (Watchlist + OnOrderRatio > 1.0)")
print("=" * 60)
print(f"Flagged: {len(compounding)} departments\n")
print(compounding.to_string(index=False))

## Cell 9 — Watchlist Overlap Summary

Cross-tab of all five layers. Departments appearing on multiple layers are
highest priority — they're underperforming across independent diagnostics,
which strengthens the signal. The on-order flag is shown as a compounding
marker (⚠) rather than a standalone layer.


In [ ]:
all_flagged = (regression_wl | dispersion_wl_set |
               tail_drag_wl_set | sell_through_wl_set)

print("=" * 60)
print("WATCHLIST OVERLAP SUMMARY")
print("=" * 60)
print(f"Total unique departments flagged: {len(all_flagged)}\n")

layer_map = {
    "Regression":   regression_wl,
    "Dispersion":   dispersion_wl_set,
    "TailDrag":     tail_drag_wl_set,
    "SellThrough":  sell_through_wl_set,
}

rows = []
for dept_name in sorted(all_flagged):
    flags  = [k for k, v in layer_map.items() if dept_name in v]
    oor    = "OnOrder⚠" if dept_name in oor_wl_set else ""
    turns  = dept.loc[dept["Department"] == dept_name, "InventoryTurns"].iloc[0]
    inv    = dept.loc[dept["Department"] == dept_name, "AvgDailyInvCOGS"].iloc[0]
    rows.append({"Department": dept_name, "Turns": turns,
                 "AvgDailyInv": inv, "Layers": len(flags),
                 "Flags": ", ".join(flags + ([oor] if oor else []))})

overlap_df = pd.DataFrame(rows).sort_values("Layers", ascending=False)
print(overlap_df.to_string(index=False))

print("\n" + "=" * 60)
print("LAYER COUNTS")
print("=" * 60)
for name, s in layer_map.items():
    print(f"  {name:<14}: {len(s)} departments")
print(f"  {'OnOrder⚠':<14}: {len(oor_wl_set)} departments (compounding, subset of above)")
print(f"  {'Total unique':<14}: {len(all_flagged)} departments")

# ── Heatmap ────────────────────────────────────────────────────────────────
heat = pd.DataFrame(
    {k: [1 if d in v else 0 for d in overlap_df["Department"]]
     for k, v in layer_map.items()},
    index=overlap_df["Department"]
)
fig, ax = plt.subplots(figsize=(8, max(4, len(all_flagged) * 0.5)))
sns.heatmap(heat, annot=True, fmt="d", cmap="RdYlGn",
            linewidths=0.5, cbar=False, ax=ax,
            vmin=0, vmax=1)
ax.set_title("Watchlist Layer Presence by Department")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

## Cell 10 — Cook's Distance

Identifies departments that disproportionately influence the regression
fit. High Cook's D on a watchlist department *strengthens* the finding —
the department is pulling the line toward itself and still underperforming.
High Cook's D on a clean department warrants manual review before trusting
the model's prediction for it.

Threshold: 4/n (conventional rule of thumb).


In [ ]:
influence  = OLSInfluence(fit)
cooks_d    = influence.cooks_distance[0]
threshold  = 4 / len(df_reg)

df_reg["CooksD"] = cooks_d
dept = dept.merge(df_reg[["Department", "CooksD"]], on="Department", how="left")

high_inf = dept[dept["CooksD"] > threshold].sort_values("CooksD", ascending=False)

print(f"Cook's D threshold (4/n = 4/{len(df_reg)}): {threshold:.4f}")
print(f"High-influence departments: {len(high_inf)}\n")
print(high_inf[["Department", "InventoryTurns", "PredictedTurns",
                "ResidualStd", "CooksD"]].round(4).to_string(index=False))

print("\nHigh-influence + watchlist overlap:")
for _, r in high_inf.iterrows():
    wl = "⚠ WATCHLIST" if r["Department"] in all_flagged else "clean"
    print(f"  {r['Department']:<28} CooksD: {r['CooksD']:.4f}  {wl}")

# ── Plot ───────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 4))
bar_colors = ["#d9534f" if d > threshold else "#4a90d9"
              for d in dept["CooksD"].fillna(0)]
ax.bar(dept["Department"], dept["CooksD"].fillna(0),
       color=bar_colors, edgecolor="white")
ax.axhline(threshold, color="#d9534f", linestyle="--", linewidth=1,
           label=f"Threshold {threshold:.3f}")
ax.set_ylabel("Cook's Distance")
ax.set_title("Cook's Distance by Department")
ax.set_xticklabels(dept["Department"], rotation=45, ha="right", fontsize=8)
ax.legend()
plt.tight_layout()
plt.show()

## Cell 11 — Diagnostic Plots

2×3 panel overview of the regression fit and watchlist structure.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

wl_colors = ["#d9534f" if d in all_flagged else "#4a90d9"
             for d in dept["Department"]]

# --- [0,0] Actual vs predicted ---
ax = axes[0, 0]
ax.scatter(dept["PredictedTurns"], dept["InventoryTurns"],
           c=wl_colors, s=80, edgecolors="white", linewidths=0.5, alpha=0.9)
lim = max(dept[["PredictedTurns","InventoryTurns"]].max()) * 1.1
ax.plot([0, lim], [0, lim], "k--", linewidth=1, alpha=0.4)
for _, r in dept[dept["Department"].isin(all_flagged)].iterrows():
    ax.annotate(r["Department"], (r["PredictedTurns"], r["InventoryTurns"]),
                fontsize=7, xytext=(4, 2), textcoords="offset points")
ax.set_xlabel("Predicted Turns"); ax.set_ylabel("Actual Turns")
ax.set_title("Actual vs. Predicted Turns")

# --- [0,1] Residuals by department ---
ax = axes[0, 1]
dept_s = dept.sort_values("ResidualStd")
rcols  = ["#d9534f" if r < -1 else "#f0ad4e" if r < 0 else "#5cb85c"
          for r in dept_s["ResidualStd"].fillna(0)]
ax.barh(dept_s["Department"], dept_s["ResidualStd"].fillna(0),
        color=rcols, edgecolor="white", height=0.7)
ax.axvline(-1, color="#d9534f", linestyle="--", linewidth=1, label="−1 std")
ax.axvline(0,  color="#555",    linestyle=":",  linewidth=0.8)
ax.set_xlabel("Standardized Residual")
ax.set_title("Regression Residuals by Department")
ax.legend(fontsize=8)
ax.tick_params(axis="y", labelsize=8)

# --- [0,2] Residuals vs dispersion ---
ax = axes[0, 2]
sc_colors = ["#d9534f" if r < -1 else "#5cb85c" if r > 1 else "#4a90d9"
             for r in dept["ResidualStd"].fillna(0)]
ax.scatter(dept["TurnsCV"], dept["ResidualStd"].fillna(0),
           c=sc_colors, s=80, edgecolors="white", linewidths=0.5, alpha=0.9)
ax.axhline(-1.0, color="#d9534f", linestyle="--", linewidth=1, label="−1 std")
ax.axhline(0,    color="#555",    linestyle=":",  linewidth=0.8)
ax.axvline(1.0,  color="#f0ad4e", linestyle=":",  linewidth=1, label="CV = 1.0")
for _, r in dept[dept["Department"].isin(all_flagged)].iterrows():
    ax.annotate(r["Department"], (r["TurnsCV"], r["ResidualStd"]),
                fontsize=6.5, xytext=(4, 2), textcoords="offset points")
ax.set_xlabel("Turns CV (within-dept dispersion)")
ax.set_ylabel("Standardized Residual")
ax.set_title("Residuals vs. Dispersion")
ax.legend(fontsize=8)

# --- [1,0] TurnsP25 vs aggregate turns ---
ax = axes[1, 0]
sc = ax.scatter(dept["TurnsP25"], dept["InventoryTurns"],
                c=dept["TurnsCV"], cmap="RdYlGn_r",
                s=80, edgecolors="white", linewidths=0.5,
                vmin=0.5, vmax=2.0, alpha=0.9)
ax.axvline(0.75, color="#d9534f", linestyle="--", linewidth=1, label="P25 = 0.75")
ax.plot([0, 5], [0, 5], "k:", linewidth=0.8, alpha=0.4)
ax.set_xlabel("Turns P25 (bottom-quartile SKUs)")
ax.set_ylabel("Aggregate Department Turns")
ax.set_title("Tail Drag — P25 vs. Aggregate Turns")
ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label="Turns CV", shrink=0.8)

# --- [1,1] STR vs markdown ---
ax = axes[1, 1]
dept_p = dept.merge(dept_markdown[["Department","MarkdownPct"]], on="Department", how="left")
ax.scatter(dept_p["SellThroughRate"], dept_p["MarkdownPct"].fillna(0),
           c=wl_colors, s=80, edgecolors="white", linewidths=0.5, alpha=0.9)
ax.axvline(0.49, color="#d9534f", linestyle="--", linewidth=1, alpha=0.7)
ax.axhline(0.20, color="#f0ad4e", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xlabel("Sell-Through Rate")
ax.set_ylabel("Markdown SKU %")
ax.set_title("Sell-Through vs. Markdown Rate")

# --- [1,2] Inventory exposure, all flagged ---
ax = axes[1, 2]
flagged_df = dept[dept["Department"].isin(all_flagged)].sort_values("AvgDailyInvCOGS")
layer_color = []
for d in flagged_df["Department"]:
    if d in regression_wl:    layer_color.append("#d9534f")
    elif d in tail_drag_wl_set: layer_color.append("#f0ad4e")
    elif d in sell_through_wl_set: layer_color.append("#9b59b6")
    else:                      layer_color.append("#5dade2")
ax.barh(flagged_df["Department"], flagged_df["AvgDailyInvCOGS"] / 1_000,
        color=layer_color, edgecolor="white", height=0.7)
ax.set_xlabel("Avg Daily Inventory at Cost ($K)")
ax.set_title("Inventory Exposure — Flagged Departments")
handles = [mpatches.Patch(color="#d9534f", label="Regression"),
           mpatches.Patch(color="#f0ad4e", label="Tail Drag"),
           mpatches.Patch(color="#9b59b6", label="Sell-Through"),
           mpatches.Patch(color="#5dade2", label="Dispersion")]
ax.legend(handles=handles, fontsize=7, loc="lower right")

plt.suptitle("Inventory Health — Diagnostic Overview", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Cell 12 — Item-Level Diagnostic

SKU detail for all watchlisted departments. Shows turns, velocity tier,
cadence, markdown status, and open PO exposure per SKU. Sorted by average
daily inventory so the highest-exposure items are at the top.

Set `TARGET_LAYERS` to limit which watchlist layers to drill into.


In [ ]:
# ── Select which layers to include ────────────────────────────────────────
TARGET_LAYERS = {
    "Regression":  regression_wl,
    "Dispersion":  dispersion_wl_set,
    "TailDrag":    tail_drag_wl_set,
    "SellThrough": sell_through_wl_set,
}

watchlist_depts = set()
for s in TARGET_LAYERS.values():
    watchlist_depts |= s

print(f"Item-level diagnostic: {len(watchlist_depts)} departments\n")

# ── SKU detail for each department ────────────────────────────────────────
diag_cols = ["Brand", "Model", "UnitsSold", "AvgDailyInvCOGS", "ItemTurns",
             "VelocityTier", "IsMarkdown", "MarkdownDepth",
             "DaysSinceLastSale", "AvgDaysBetweenSales", "MaxGapDays",
             "HasOpenPO", "UnitsOnOrder", "POCostValue"]

for dept_name in sorted(watchlist_depts):
    dept_skus = (skus[skus["Department"] == dept_name]
                 [diag_cols + ["SKUID"]]
                 .sort_values("AvgDailyInvCOGS", ascending=False))
    dept_row  = dept[dept["Department"] == dept_name].iloc[0]
    flags     = [k for k, v in TARGET_LAYERS.items() if dept_name in v]

    print("=" * 70)
    print(f"DEPARTMENT: {dept_name}  [{', '.join(flags)}]")
    print("=" * 70)
    print(f"  Turns: {dept_row['InventoryTurns']:.3f}  |  "
          f"Predicted: {dept_row.get('PredictedTurns', float('nan')):.3f}  |  "
          f"ResidualStd: {dept_row.get('ResidualStd', float('nan')):.3f}  |  "
          f"STR: {dept_row.get('SellThroughRate', float('nan')):.3f}  |  "
          f"TurnsCV: {dept_row['TurnsCV']:.3f}  |  "
          f"P25: {dept_row['TurnsP25']:.3f}")
    print(f"  SKUs: {dept_row['SKUCount']}  |  "
          f"Brands: {dept_row['BrandCount']}  |  "
          f"AvgDailyInv: ${dept_row['AvgDailyInvCOGS']:,.0f}  |  "
          f"OnOrderCOGS: ${dept_row.get('TotalOnOrderCOGS', 0):,.0f}  |  "
          f"OnOrderRatio: {dept_row.get('OnOrderRatio', 0):.2f}x")
    print()

    with pd.option_context("display.max_columns", None, "display.width", 220,
                           "display.max_colwidth", 40,
                           "display.float_format", "{:,.1f}".format):
        print(dept_skus.drop(columns=["SKUID"]).to_string(index=False))

    # ── Velocity tier summary ──────────────────────────────────────────────
    tier_order = ["Dead", "Slow", "Moderate", "Fast"]
    tier_sum   = (dept_skus.groupby("VelocityTier")
                  .agg(SKUs=("SKUID","count"),
                       TotalDailyInv=("AvgDailyInvCOGS","sum"))
                  .reindex(tier_order)
                  .dropna(how="all"))
    total_inv  = tier_sum["TotalDailyInv"].sum()
    print(f"\n  Velocity Tier Breakdown:")
    print(f"  {'Tier':<12} {'SKUs':>6} {'DailyInv':>12} {'% of Dept':>10}")
    print(f"  {'-'*44}")
    for tier, row in tier_sum.iterrows():
        if pd.isna(row["SKUs"]): continue
        pct = row["TotalDailyInv"] / total_inv * 100
        print(f"  {tier:<12} {int(row['SKUs']):>6} "
              f"${row['TotalDailyInv']:>11,.0f} {pct:>9.1f}%")
    print(f"  {'TOTAL':<12} {int(tier_sum['SKUs'].sum()):>6} "
          f"${total_inv:>11,.0f} {'100.0%':>10}")
    print()

## Cell 13 — Item Velocity Flags

Applies cadence thresholds at the SKU level across all watchlisted
departments. Each flagged SKU gets a plain-English flag string describing
which thresholds it tripped.

Adjust the three threshold constants to change sensitivity.


In [ ]:
DAYS_SINCE_THRESHOLD = 90
MAX_GAP_THRESHOLD    = 60
AVG_GAP_THRESHOLD    = 30

wl_skus = skus[skus["Department"].isin(watchlist_depts)].copy()

velocity_flags = wl_skus[
    (wl_skus["DaysSinceLastSale"]   > DAYS_SINCE_THRESHOLD) |
    (wl_skus["MaxGapDays"]          > MAX_GAP_THRESHOLD)    |
    (wl_skus["AvgDaysBetweenSales"] > AVG_GAP_THRESHOLD)
].copy()

def build_flag(r):
    parts = []
    if pd.notna(r["DaysSinceLastSale"])   and r["DaysSinceLastSale"]   > DAYS_SINCE_THRESHOLD:
        parts.append(f"No sale {int(r['DaysSinceLastSale'])}d")
    if pd.notna(r["MaxGapDays"])          and r["MaxGapDays"]          > MAX_GAP_THRESHOLD:
        parts.append(f"Max gap {int(r['MaxGapDays'])}d")
    if pd.notna(r["AvgDaysBetweenSales"]) and r["AvgDaysBetweenSales"] > AVG_GAP_THRESHOLD:
        parts.append(f"Avg gap {r['AvgDaysBetweenSales']:.0f}d")
    return ", ".join(parts)

velocity_flags["VelocityFlag"] = velocity_flags.apply(build_flag, axis=1)

flag_cols = ["Department", "Brand", "Model", "AvgDailyInvCOGS", "ItemTurns",
             "VelocityTier", "DaysSinceLastSale", "AvgDaysBetweenSales",
             "MaxGapDays", "IsMarkdown", "HasOpenPO", "VelocityFlag"]

velocity_flags = (velocity_flags[flag_cols]
                  .sort_values(["Department","DaysSinceLastSale"], ascending=[True, False]))

print("=" * 60)
print("ITEM VELOCITY FLAGS — Watchlist Departments")
print("=" * 60)
print(f"Thresholds: Days since last sale > {DAYS_SINCE_THRESHOLD}  |  "
      f"Max gap > {MAX_GAP_THRESHOLD}  |  Avg gap > {AVG_GAP_THRESHOLD}")
print(f"Flagged SKUs: {len(velocity_flags):,}\n")

for dept_name in sorted(watchlist_depts):
    dept_flags = velocity_flags[velocity_flags["Department"] == dept_name]
    if dept_flags.empty:
        continue
    print(f"\n{'='*60}")
    print(f"  {dept_name}  ({len(dept_flags)} flagged SKUs)")
    print(f"{'='*60}")
    with pd.option_context("display.max_columns", None, "display.width", 220,
                           "display.max_colwidth", 40,
                           "display.float_format", "{:,.1f}".format):
        print(dept_flags.drop(columns=["Department"]).to_string(index=False))